<a href="https://colab.research.google.com/github/Di-oss/com4/blob/%D0%A1%D0%B0%D1%80%D0%BD%D0%B0%D0%B2%D1%81%D0%BA%D0%B0%D1%8F/dev_sarnavskaya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Модуль 2: Список желаний
# Интернет-магазин "Все и сразу"

from datetime import datetime
import uuid

class Wishlist:
    """Класс списка желаний"""
    def __init__(self, customer_id, customer_name, name="Избранное"):
        self.id = str(uuid.uuid4())[:8]
        self.customer_id = customer_id
        self.customer_name = customer_name
        self.name = name
        self.created_at = datetime.now()
        self.products = []  # Список ID товаров
        self.is_public = False

    def add_product(self, product_id, product_name):
        if product_id not in self.products:
            self.products.append({
                'id': product_id,
                'name': product_name,
                'added_at': datetime.now()
            })
            print(f"Товар {product_name} добавлен в {self.name}")
            return True
        print("Товар уже в списке")
        return False

    def remove_product(self, product_id):
        for i, p in enumerate(self.products):
            if p['id'] == product_id:
                removed = self.products.pop(i)
                print(f"Товар {removed['name']} удален из {self.name}")
                return True
        return False

    def toggle_public(self):
        self.is_public = not self.is_public
        status = "открыт" if self.is_public else "закрыт"
        print(f"Список теперь {status}")

    def get_info(self):
        info = f"\n=== {self.name} ===\n"
        info += f"Владелец: {self.customer_name}\n"
        info += f"Создан: {self.created_at.strftime('%d.%m.%Y')}\n"
        info += f"Товаров: {len(self.products)}\n"
        info += "--- Товары ---\n"
        for i, p in enumerate(self.products, 1):
            info += f"{i}. {p['name']} (добавлен {p['added_at'].strftime('%d.%m.%Y')})\n"
        info += "="*30
        return info

class WishlistManager:
    """Менеджер списков желаний"""
    def __init__(self):
        self.wishlists = {}  # id -> Wishlist
        self.customer_wishlists = {}  # customer_id -> [wishlist_ids]

    def create_wishlist(self, customer_id, customer_name, name="Избранное"):
        wishlist = Wishlist(customer_id, customer_name, name)
        self.wishlists[wishlist.id] = wishlist

        if customer_id not in self.customer_wishlists:
            self.customer_wishlists[customer_id] = []
        self.customer_wishlists[customer_id].append(wishlist.id)

        print(f"Список '{name}' создан (ID: {wishlist.id})")
        return wishlist

    def get_customer_wishlists(self, customer_id):
        wishlist_ids = self.customer_wishlists.get(customer_id, [])
        return [self.wishlists[wid] for wid in wishlist_ids if wid in self.wishlists]

    def get_public_wishlists(self):
        return [w for w in self.wishlists.values() if w.is_public]

    def add_to_wishlist(self, wishlist_id, product_id, product_name):
        wishlist = self.wishlists.get(wishlist_id)
        if wishlist:
            return wishlist.add_product(product_id, product_name)
        return False

    def remove_from_wishlist(self, wishlist_id, product_id):
        wishlist = self.wishlists.get(wishlist_id)
        if wishlist:
            return wishlist.remove_product(product_id)
        return False

    def share_wishlist(self, wishlist_id):
        wishlist = self.wishlists.get(wishlist_id)
        if wishlist:
            wishlist.toggle_public()
            if wishlist.is_public:
                print(f"Ссылка для просмотра: shop.ru/wishlist/{wishlist.id}")

# Тестовые данные
manager = WishlistManager()

# Главный цикл
while True:
    print("\n" + "="*50)
    print("МОДУЛЬ СПИСКА ЖЕЛАНИЙ")
    print("="*50)
    print("1 - Создать список")
    print("2 - Мои списки")
    print("3 - Добавить товар")
    print("4 - Удалить товар")
    print("5 - Открытые списки")
    print("6 - Поделиться списком")
    print("0 - Выход")

    choice = input("Выберите: ")

    if choice == "1":
        cust_id = input("Ваш ID: ")
        name = input("Ваше имя: ")
        list_name = input("Название списка (Enter - Избранное): ") or "Избранное"
        manager.create_wishlist(cust_id, name, list_name)

    elif choice == "2":
        cust_id = input("Ваш ID: ")
        lists = manager.get_customer_wishlists(cust_id)
        if lists:
            for w in lists:
                print(w.get_info())
        else:
            print("У вас нет списков")

    elif choice == "3":
        cust_id = input("Ваш ID: ")
        lists = manager.get_customer_wishlists(cust_id)
        if not lists:
            print("Сначала создайте список")
            continue

        print("Ваши списки:")
        for i, w in enumerate(lists, 1):
            print(f"{i}. {w.name}")

        try:
            num = int(input("Номер списка: ")) - 1
            wishlist = lists[num]
            prod_id = input("ID товара: ")
            prod_name = input("Название товара: ")
            manager.add_to_wishlist(wishlist.id, prod_id, prod_name)
        except:
            print("Ошибка")

    elif choice == "4":
        cust_id = input("Ваш ID: ")
        lists = manager.get_customer_wishlists(cust_id)
        if not lists:
            continue

        for i, w in enumerate(lists, 1):
            print(f"{i}. {w.name}")

        try:
            num = int(input("Номер списка: ")) - 1
            wishlist = lists[num]
            wishlist.get_info()
            prod_id = input("ID товара для удаления: ")
            manager.remove_from_wishlist(wishlist.id, prod_id)
        except:
            print("Ошибка")

    elif choice == "5":
        public = manager.get_public_wishlists()
        if public:
            print("\n=== ОТКРЫТЫЕ СПИСКИ ===")
            for w in public:
                print(f"{w.name} от {w.customer_name} ({len(w.products)} товаров)")
        else:
            print("Нет открытых списков")

    elif choice == "6":
        cust_id = input("Ваш ID: ")
        lists = manager.get_customer_wishlists(cust_id)
        if lists:
            for i, w in enumerate(lists, 1):
                print(f"{i}. {w.name}")
            try:
                num = int(input("Номер списка: ")) - 1
                manager.share_wishlist(lists[num].id)
            except:
                print("Ошибка")

    elif choice == "0":
        break


МОДУЛЬ СПИСКА ЖЕЛАНИЙ
1 - Создать список
2 - Мои списки
3 - Добавить товар
4 - Удалить товар
5 - Открытые списки
6 - Поделиться списком
0 - Выход
Выберите: 5
Нет открытых списков

МОДУЛЬ СПИСКА ЖЕЛАНИЙ
1 - Создать список
2 - Мои списки
3 - Добавить товар
4 - Удалить товар
5 - Открытые списки
6 - Поделиться списком
0 - Выход
Выберите: 2
Ваш ID: 1
У вас нет списков

МОДУЛЬ СПИСКА ЖЕЛАНИЙ
1 - Создать список
2 - Мои списки
3 - Добавить товар
4 - Удалить товар
5 - Открытые списки
6 - Поделиться списком
0 - Выход
Выберите: 1
Ваш ID: 2
Ваше имя: Вкуе
Название списка (Enter - Избранное): Штучки
Список 'Штучки' создан (ID: be93fcbc)

МОДУЛЬ СПИСКА ЖЕЛАНИЙ
1 - Создать список
2 - Мои списки
3 - Добавить товар
4 - Удалить товар
5 - Открытые списки
6 - Поделиться списком
0 - Выход
Выберите: 3
Ваш ID: 2
Ваши списки:
1. Штучки
Номер списка: 1
ID товара: 5
Название товара: кукла
Товар кукла добавлен в Штучки

МОДУЛЬ СПИСКА ЖЕЛАНИЙ
1 - Создать список
2 - Мои списки
3 - Добавить товар
4 - Удалить товар